### Transformação de dados na Camada Gold

Leitura dos dados da camada Silver, aplicação das regras de negócio KPI

In [0]:
# CARREGAMENTO DE VARIÁVEIS DE AMBIENTE
import os
from datetime import datetime
from dotenv import load_dotenv
from pyspark.sql.functions import col, current_timestamp, year, month, trim, when, lit, row_number, to_date, to_timestamp,coalesce
from pyspark.sql.functions import trunc,count,sum,avg,round,date_trunc,row_number, coalesce,lag, datediff
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType, TimestampType
from delta.tables import DeltaTable


load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

jdbc_host = os.getenv("JDBC_HOST")
jdbc_database = os.getenv("JDBC_DATABASE")
jdbc_username = os.getenv("JDBC_USERNAME")
jdbc_password = os.getenv("JDBC_PASSWORD")

# JDBC CONFIGS
jdbc_url = (
    f"jdbc:sqlserver://{jdbc_host}:1433;database={jdbc_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)
jdbc_properties = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

schema_name = "squad3"

# DICIONÁRIO DE CREDENCIAIS OAUTH PARA USAR COM SPARK
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# DICIONÁRIO DE PATHS
paths = {
    "food_avaliacoes_produto_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_avaliacoes_produto",
    "food_estoque_lojas_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_estoque_lojas",
    "food_avaliacoes_produto_gold": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_avaliacoes_produto",
    "food_estoque_lojas_gold": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_estoque_lojas",
    "raw_path": f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data",
    "gold_avaliacoes_antes_pedido": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_avaliacoes_produto/gold_avaliacoes_antes_pedido",
    "gold_kpi_nota_media_sku_mes": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_avaliacoes_produto/gold_kpi_nota_media_sku_mes",
    "gold_kpi_status_avaliacoes_mes": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_avaliacoes_produto/gold_kpi_status_avaliacoes_mes",
    "gold_kpi_top10_piores_skus": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_avaliacoes_produto/gold_kpi_top10_piores_skus",
    "gold_avaliacoes_produtos_enriquecidas": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_avaliacoes_produto/gold_avaliacoes_produtos_enriquecidas",
    "gold_kpi_ruptura_estoque_mes": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_estoque_lojas/gold_kpi_ruptura_estoque_mes",
    "gold_kpi_lotes_vencidos_por_loja": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_estoque_lojas/gold_kpi_lotes_vencidos_por_loja",
    "gold_giro_estoque_por_categoria_pai": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_estoque_lojas/gold_giro_estoque_por_categoria_pai","gold_alerta_estoque_negativo": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_estoque_lojas/gold_alerta_estoque_negativo",
    "gold_ruptura_estoque_enriquecida": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/gold/food_estoque_lojas/gold_ruptura_estoque_enriquecida",
}



In [0]:
# LEITURA DAS TABELAS SILVER
df_silver_avaliacoes = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_avaliacoes_produto_silver"])
)

df_silver_estoque = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_estoque_lojas_silver"])
)

# LEITURA DAS TABELAS DE APOIO
df_ecommerce_pedidos = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_pedidos.csv")
)

df_ecommerce_itens_pedido = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_itens_pedido.csv")
)

df_food_lotes_producao = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/food_lotes_producao.csv")
)

df_food_fornecedores = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/food_fornecedores.csv")
)

df_physical_lojas = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_lojas.csv")
)

df_physical_produtos_pereciveis = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_produtos_pereciveis.csv")
)

df_ecommerce_categorias = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_categorias.csv")
)

df_ecommerce_produtos = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_produtos.csv")
)

df_physical_itens_venda_caixa = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_itens_venda_caixa.csv")
)

df_physical_vendas_caixa = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_vendas_caixa.csv")
)


In [0]:
# GOLD - AVALIAÇÕES + ESTOQUE
# Adição do gold_ingested_at

from datetime import datetime
from pyspark.sql.functions import col,lit,trunc,count,sum,avg,round,date_trunc,row_number,when,coalesce,lag,datediff
from pyspark.sql.window import Window


# ============================================================
# TIMESTAMP DA EXECUÇÃO GOLD
# ============================================================

gold_ingested_at = datetime.now()


# ============================================================
# ============================================================
#                 FOOD AVALIAÇÕES PRODUTO
# ============================================================
# ============================================================


# ============================================================
# REGRA 6
# dt_avaliacao não pode ser anterior à dt_pedido
# ============================================================

df_gold_avaliacoes_antes_pedido = (
    df_silver_avaliacoes.alias("a")
    .join(
        df_ecommerce_pedidos
        .select(
            col("id_pedido").alias("id_pedido_ref"),
            col("dt_pedido")
        )
        .alias("p"),
        col("a.id_pedido") == col("p.id_pedido_ref"),
        "left"
    )
    .withColumn(
        "flag_avaliacao_antes_pedido",
        when(
            (col("a.verificada") == True)
            & (
                col("a.flag_avaliacao_pedido_verificado_invalido")
                == False
            )
            & (
                col("a.dt_avaliacao") < col("p.dt_pedido")
            ),
            lit(True)
        ).otherwise(lit(False))
    )
    .select(
        col("a.id_avaliacao"),
        col("a.id_pedido"),
        col("a.verificada"),
        col("a.dt_avaliacao"),
        col("flag_avaliacao_antes_pedido")
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_avaliacoes_antes_pedido
    .limit(5)
)


# ============================================================
# REGRA 7
# KPI: nota média por SKU por mês
# ============================================================

df_gold_kpi_nota_media_sku_mes = (
    df_silver_avaliacoes
    .select(
        col("sku"),
        col("nota"),
        col("dt_avaliacao")
    )
    .withColumn(
        "mes_avaliacao",
        trunc(
            col("dt_avaliacao"),
            "month"
        )
    )
    .groupBy(
        "sku",
        "mes_avaliacao"
    )
    .agg(
        avg("nota").alias("nota_media")
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_kpi_nota_media_sku_mes
    .limit(5)
)


# ============================================================
# REGRA 8
# KPI: % avaliações verificadas vs não verificadas por mês
# ============================================================

df_gold_kpi_status_avaliacoes_mes = (
    df_silver_avaliacoes
    .select(
        col("verificada"),
        col("dt_avaliacao")
    )
    .withColumn(
        "mes_avaliacao",
        trunc(
            col("dt_avaliacao"),
            "month"
        )
    )
    .groupBy(
        "mes_avaliacao"
    )
    .agg(
        count("*").alias(
            "qtd_avaliacoes"
        ),

        sum(
            when(
                col("verificada") == True,
                1
            ).otherwise(0)
        ).alias(
            "qtd_avaliacoes_verificadas"
        ),

        sum(
            when(
                col("verificada") == False,
                1
            ).otherwise(0)
        ).alias(
            "qtd_avaliacoes_nao_verificadas"
        )
    )
    .withColumn(
        "pct_avaliacoes_verificadas",
        round(
            col("qtd_avaliacoes_verificadas")
            / col("qtd_avaliacoes") * 100,
            2
        )
    )
    .withColumn(
        "pct_avaliacoes_nao_verificadas",
        round(
            col("qtd_avaliacoes_nao_verificadas")
            / col("qtd_avaliacoes") * 100,
            2
        )
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_kpi_status_avaliacoes_mes
    .limit(5)
)


# ============================================================
# REGRA 9
# KPI: Top 10 SKUs com pior nota média por trimestre
# ============================================================

df_gold_kpi_top10_piores_skus = (
    df_silver_avaliacoes
    .select(
        col("sku"),
        col("nota"),
        col("dt_avaliacao")
    )
    .withColumn(
        "trimestre_avaliacao",
        date_trunc(
            "quarter",
            col("dt_avaliacao")
        )
    )
    .groupBy(
        "sku",
        "trimestre_avaliacao"
    )
    .agg(
        count("*").alias(
            "qtd_avaliacoes"
        ),

        round(
            avg("nota"),
            2
        ).alias(
            "nota_media"
        )
    )
    .filter(
        col("qtd_avaliacoes") >= 5
    )
)

window_top10 = (
    Window
    .partitionBy(
        "trimestre_avaliacao"
    )
    .orderBy(
        col("nota_media").asc()
    )
)

df_gold_kpi_top10_piores_skus = (
    df_gold_kpi_top10_piores_skus
    .withColumn(
        "ranking",
        row_number().over(
            window_top10
        )
    )
    .filter(
        col("ranking") <= 10
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_kpi_top10_piores_skus
    .orderBy(
        "trimestre_avaliacao",
        "ranking"
    )
)


# ============================================================
# REGRA 10
# Enriquecimento com lote e fornecedor
# ============================================================

df_gold_avaliacoes_produtos_enriquecidas = (
    df_silver_avaliacoes.alias("a")

    # Avaliação -> item do pedido
    .join(
        df_ecommerce_itens_pedido
        .select(
            col("id_pedido").alias(
                "id_pedido_item"
            ),
            col("sku").alias(
                "sku_item"
            )
        )
        .alias("i"),

        (
            (col("a.id_pedido") ==
             col("i.id_pedido_item"))
            &
            (col("a.sku") ==
             col("i.sku_item"))
        ),

        "left"
    )

    # Item -> lote pelo SKU
    .join(
        df_food_lotes_producao
        .select(
            col("id_lote"),
            col("sku").alias(
                "sku_lote"
            ),
            col("id_fornecedor")
        )
        .alias("l"),

        col("i.sku_item") ==
        col("l.sku_lote"),

        "left"
    )

    # Lote -> fornecedor
    .join(
        df_food_fornecedores
        .select(
            col("id_fornecedor").alias(
                "id_fornecedor_ref"
            ),
            col("razao_social").alias(
                "fornecedor"
            )
        )
        .alias("f"),

        col("l.id_fornecedor") ==
        col("f.id_fornecedor_ref"),

        "left"
    )

    .select(
        col("a.id_avaliacao"),
        col("a.id_pedido"),
        col("a.sku"),
        col("a.nota"),
        col("a.dt_avaliacao"),
        col("a.verificada"),
        col("l.id_lote"),
        col("l.id_fornecedor"),
        col("f.fornecedor")
    )

    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_avaliacoes_produtos_enriquecidas
    .limit(10)
)


# ============================================================
# ============================================================
#                    FOOD ESTOQUE LOJAS
# ============================================================
# ============================================================


# ============================================================
# MAPA DE CATEGORIAS
# Necessário para a Regra 8
# ============================================================

df_categorias_pereciveis = (
    df_physical_produtos_pereciveis
    .select(
        col("sku"),
        col("categoria_pai")
    )
    .withColumn(
        "tipo_produto",
        lit("Perecível")
    )
)


df_categorias_secos = (
    df_ecommerce_produtos
    .select(
        col("sku"),
        col("id_categoria")
    )
    .join(
        df_ecommerce_categorias
        .select(
            col("id_categoria").alias(
                "id_categoria_ref"
            ),
            col("nome_categoria_pai")
        )
        .dropDuplicates(
            ["id_categoria_ref"]
        )
        .alias("c"),

        col("id_categoria") ==
        col("id_categoria_ref"),

        "left"
    )
    .select(
        col("sku"),
        col("nome_categoria_pai").alias(
            "categoria_pai"
        )
    )
    .withColumn(
        "tipo_produto",
        lit("Seco")
    )
)


df_mapa_categoria_sku = (
    df_categorias_pereciveis
    .unionByName(
        df_categorias_secos
    )
    .dropDuplicates(
        ["sku"]
    )
)


# ============================================================
# REGRA 6
# KPI: Taxa de ruptura de estoque por loja por mês
# ============================================================

df_gold_kpi_ruptura_estoque_mes = (
    df_silver_estoque
    .select(
        col("id_loja"),
        col("quantidade_disponivel"),
        col("estoque_minimo"),
        col("dt_snapshot")
    )
    .join(
        df_physical_lojas
        .select(
            col("id_loja").alias(
                "id_loja_ref"
            ),
            col("nome_loja")
        )
        .alias("l"),

        col("id_loja") ==
        col("id_loja_ref"),

        "left"
    )
    .withColumn(
        "mes_snapshot",
        trunc(
            col("dt_snapshot"),
            "month"
        )
    )
    .groupBy(
        "id_loja",
        "nome_loja",
        "mes_snapshot"
    )
    .agg(
        count("*").alias(
            "qtd_snapshots"
        ),

        sum(
            when(
                col("quantidade_disponivel")
                < col("estoque_minimo"),
                1
            ).otherwise(0)
        ).alias(
            "qtd_snapshots_ruptura"
        )
    )
    .withColumn(
        "taxa_ruptura_pct",
        round(
            col("qtd_snapshots_ruptura")
            / col("qtd_snapshots") * 100,
            2
        )
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_kpi_ruptura_estoque_mes
    .limit(10)
)


# ============================================================
# REGRA 7
# KPI: % snapshots de perecíveis vinculados a lote vencido
# por loja
# ============================================================

df_gold_kpi_lotes_vencidos_por_loja = (
    df_silver_estoque
    .select(
        col("id_loja"),
        col("id_lote"),
        col("dt_snapshot")
    )
    .join(
        df_food_lotes_producao
        .select(
            col("id_lote").alias(
                "id_lote_ref"
            ),
            col("dt_validade")
        )
        .dropDuplicates(
            ["id_lote_ref"]
        )
        .alias("l"),

        col("id_lote") ==
        col("id_lote_ref"),

        "left"
    )
    .join(
        df_physical_lojas
        .select(
            col("id_loja").alias(
                "id_loja_ref"
            ),
            col("nome_loja")
        )
        .dropDuplicates(
            ["id_loja_ref"]
        )
        .alias("loja"),

        col("id_loja") ==
        col("id_loja_ref"),

        "left"
    )
    .withColumn(
        "mes_snapshot",
        trunc(
            col("dt_snapshot"),
            "month"
        )
    )
    .groupBy(
        "id_loja",
        "nome_loja",
        "mes_snapshot"
    )
    .agg(
        count("*").alias(
            "qtd_snapshots"
        ),

        sum(
            when(
                col("dt_snapshot") >
                col("dt_validade"),
                1
            ).otherwise(0)
        ).alias(
            "qtd_snapshots_lote_vencido"
        )
    )
    .withColumn(
        "pct_snapshots_lote_vencido",
        round(
            col("qtd_snapshots_lote_vencido")
            / col("qtd_snapshots") * 100,
            2
        )
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_kpi_lotes_vencidos_por_loja

    .limit(10)
)


# ============================================================
# REGRA 8
# KPI: Giro médio de estoque por categoria_pai
# ============================================================

df_gold_giro_base = (
    df_silver_estoque
    .select(
        col("id_loja"),
        col("sku"),
        col("quantidade_disponivel"),
        col("dt_snapshot")
    )
)

window_estoque = (
    Window
    .partitionBy(
        "id_loja",
        "sku"
    )
    .orderBy(
        "dt_snapshot"
    )
)

df_gold_giro_base = (
    df_gold_giro_base
    .withColumn(
        "quantidade_anterior",
        lag(
            "quantidade_disponivel"
        ).over(window_estoque)
    )
    .withColumn(
        "dt_snapshot_anterior",
        lag(
            "dt_snapshot"
        ).over(window_estoque)
    )
)

df_gold_giro_base = (
    df_gold_giro_base
    .filter(
        col(
            "quantidade_anterior"
        ).isNotNull()
        &
        (
            col("quantidade_disponivel")
            <
            col("quantidade_anterior")
        )
    )
    .withColumn(
        "intervalo_dias",
        datediff(
            col("dt_snapshot"),
            col("dt_snapshot_anterior")
        )
    )
)

df_gold_giro_estoque_por_categoria_pai = (
    df_gold_giro_base.alias("e")
    .join(
        df_mapa_categoria_sku.alias("cat"),

        col("e.sku") ==
        col("cat.sku"),

        "left"
    )
    .select(
        col("e.id_loja"),
        col("e.sku"),
        col("e.intervalo_dias"),
        col("cat.categoria_pai"),
        col("cat.tipo_produto")
    )
    .groupBy(
        "tipo_produto",
        "categoria_pai"
    )
    .agg(
        round(
            avg("intervalo_dias"),
            2
        ).alias(
            "giro_medio_dias"
        )
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_giro_estoque_por_categoria_pai
    .orderBy(
        "tipo_produto",
        "categoria_pai"
    )
    .limit(10)
)


# ============================================================
# REGRA 9
# ALERTA: quantidade_disponivel negativa que não foi
# sinalizada pelo Silver
# ============================================================

df_gold_alerta_estoque_negativo = (
    df_silver_estoque
    .select(
        col("id_loja"),
        col("sku"),
        col("id_lote"),
        col("quantidade_disponivel"),
        col("dt_snapshot"),
        col("flag_quantidade_invalida")
    )
    .filter(
        (col("quantidade_disponivel") < 0)
        &
        (
            col("flag_quantidade_invalida")
            == False
        )
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_alerta_estoque_negativo
    .limit(10)
)


# ============================================================
# REGRA 10
# Validar se ruptura de estoque coincide com queda nas vendas
# ============================================================

df_base_estoque = (
    df_silver_estoque
    .select(
        col("id_loja"),
        col("sku"),
        col("quantidade_disponivel"),
        col("estoque_minimo"),
        col("dt_snapshot")
    )
)

df_itens_venda = (
    df_physical_itens_venda_caixa
    .select(
        col("id_transacao"),
        col("codigo_barras_produto").alias(
            "sku"
        ),
        col("quantidade")
    )
)

df_vendas = (
    df_itens_venda.alias("i")
    .join(
        df_physical_vendas_caixa
        .select(
            col("id_transacao"),
            col("id_loja"),
            col("dt_venda")
        )
        .alias("v"),

        col("i.id_transacao") ==
        col("v.id_transacao"),

        "inner"
    )
    .select(
        col("v.id_loja"),
        col("i.sku"),
        col("v.dt_venda"),
        col("i.quantidade")
    )
)

df_vendas_diarias = (
    df_vendas
    .withColumn(
        "dt_venda",
        col("dt_venda").cast("date")
    )
    .groupBy(
        "id_loja",
        "sku",
        "dt_venda"
    )
    .agg(
        sum("quantidade").alias(
            "quantidade_vendida"
        )
    )
)

df_gold_regra10 = (
    df_base_estoque.alias("e")
    .join(
        df_vendas_diarias.alias("v"),

        (
            (col("e.id_loja") ==
             col("v.id_loja"))
            &
            (col("e.sku") ==
             col("v.sku"))
            &
            (col("e.dt_snapshot") ==
             col("v.dt_venda"))
        ),

        "left"
    )
    .select(
        col("e.id_loja"),
        col("e.sku"),
        col("e.dt_snapshot"),
        col("e.quantidade_disponivel"),
        col("e.estoque_minimo"),

        coalesce(
            col("v.quantidade_vendida"),
            lit(0)
        ).alias(
            "quantidade_vendida"
        )
    )
)

window_vendas = (
    Window
    .partitionBy(
        "id_loja",
        "sku"
    )
    .orderBy(
        "dt_snapshot"
    )
)

df_gold_ruptura_estoque_enriquecida = (
    df_gold_regra10
    .withColumn(
        "quantidade_vendida_anterior",
        lag(
            "quantidade_vendida"
        ).over(window_vendas)
    )
    .withColumn(
        "flag_ruptura_coincide_queda_vendas",
        (
            (
                col("quantidade_disponivel")
                <
                col("estoque_minimo")
            )
            &
            col(
                "quantidade_vendida_anterior"
            ).isNotNull()
            &
            (
                col("quantidade_vendida")
                <
                col("quantidade_vendida_anterior")
            )
        )
    )
    .select(
        "id_loja",
        "sku",
        "dt_snapshot",
        "quantidade_disponivel",
        "estoque_minimo",
        "quantidade_vendida",
        "quantidade_vendida_anterior",
        "flag_ruptura_coincide_queda_vendas"
    )
    .withColumn(
        "gold_ingested_at",
        lit(gold_ingested_at)
    )
)

display(
    df_gold_ruptura_estoque_enriquecida
    .orderBy(
        "id_loja",
        "sku",
        "dt_snapshot"
    )
    .limit(10)
)

In [0]:
print("df_gold_avaliacoes_antes_pedido colunas:")
df_gold_avaliacoes_antes_pedido.printSchema()

print("df_gold_kpi_nota_media_sku_mes colunas:")
df_gold_kpi_nota_media_sku_mes.printSchema()

print("df_gold_kpi_status_avaliacoes_mes colunas:")
df_gold_kpi_status_avaliacoes_mes.printSchema()

print("df_gold_kpi_top10_piores_skus colunas:")
df_gold_kpi_top10_piores_skus.printSchema()

print("df_gold_avaliacoes_produtos_enriquecidas colunas:")
df_gold_avaliacoes_produtos_enriquecidas.printSchema()

print("df_gold_kpi_ruptura_estoque_mes colunas:")
df_gold_kpi_ruptura_estoque_mes.printSchema()

print("df_gold_kpi_lotes_vencidos_por_loja colunas:")
df_gold_kpi_lotes_vencidos_por_loja.printSchema()

print("df_gold_giro_estoque_por_categoria_pai colunas:")
df_gold_giro_estoque_por_categoria_pai.printSchema()

print("df_gold_alerta_estoque_negativo colunas:")
df_gold_alerta_estoque_negativo.printSchema()

print("df_gold_ruptura_estoque_enriquecida colunas:")
df_gold_ruptura_estoque_enriquecida.printSchema()

In [0]:
# GRAVAÇÃO GOLD - FOOD_AVALIACOES_PRODUTO
# ============================================================
# GRAVAÇÃO GOLD - FOOD AVALIAÇÕES PRODUTO
# REGRAS 6 A 10
# Upsert lógico via LEFT ANTI + UNION
# ============================================================


# ============================================================
# REGRA 6
# dt_avaliacao não pode ser anterior à dt_pedido
# Chave: id_avaliacao
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_avaliacoes_antes_pedido"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_avaliacoes_antes_pedido

    print("Gold da Regra 6 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_avaliacoes_antes_pedido
        .select("id_avaliacao")
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on="id_avaliacao",
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_avaliacoes_antes_pedido
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 6 já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_avaliacoes_antes_pedido"])
)

print("✅ Gold da Regra 6 gravada com sucesso.")


# ============================================================
# REGRA 7
# KPI: nota média por SKU por mês
# Chave: sku + mes_avaliacao
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_kpi_nota_media_sku_mes"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_kpi_nota_media_sku_mes

    print("Gold da Regra 7 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_kpi_nota_media_sku_mes
        .select(
            "sku",
            "mes_avaliacao"
        )
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on=[
                "sku",
                "mes_avaliacao"
            ],
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_kpi_nota_media_sku_mes
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 7 já existe.")
    print("Meses/SKUs afetados foram recalculados.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_kpi_nota_media_sku_mes"])
)

print("✅ Gold da Regra 7 gravada com sucesso.")


# ============================================================
# REGRA 8
# KPI: % avaliações verificadas vs não verificadas por mês
# Chave: mes_avaliacao
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_kpi_status_avaliacoes_mes"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_kpi_status_avaliacoes_mes

    print("Gold da Regra 8 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_kpi_status_avaliacoes_mes
        .select("mes_avaliacao")
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on="mes_avaliacao",
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_kpi_status_avaliacoes_mes
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 8 já existe.")
    print("Meses afetados foram recalculados.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_kpi_status_avaliacoes_mes"])
)

print("✅ Gold da Regra 8 gravada com sucesso.")


# ============================================================
# REGRA 9
# KPI: Top 10 SKUs com pior nota média por trimestre
# Unidade de atualização: trimestre inteiro
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_kpi_top10_piores_skus"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_kpi_top10_piores_skus

    print("Gold da Regra 9 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    # Remove da Gold os trimestres que foram recalculados.
    # Isso é necessário porque um SKU pode sair do Top 10.
    chaves_gold = (
        df_gold_kpi_top10_piores_skus
        .select("trimestre_avaliacao")
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on="trimestre_avaliacao",
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_kpi_top10_piores_skus
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 9 já existe.")
    print("Trimestres afetados foram recalculados.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_kpi_top10_piores_skus"])
)

print("✅ Gold da Regra 9 gravada com sucesso.")


# ============================================================
# REGRA 10
# Enriquecimento com lote e fornecedor
# Chave: id_avaliacao + id_lote
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_avaliacoes_produtos_enriquecidas"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_avaliacoes_produtos_enriquecidas

    print("Gold da Regra 10 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_avaliacoes_produtos_enriquecidas
        .select(
            "id_avaliacao",
            "id_lote"
        )
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on=[
                "id_avaliacao",
                "id_lote"
            ],
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_avaliacoes_produtos_enriquecidas
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 10 já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_avaliacoes_produtos_enriquecidas"])
)

print("✅ Gold da Regra 10 gravada com sucesso.")


print("============================================================")
print("✅ REGRAS GOLD 6 A 10 GRAVADAS COM SUCESSO.")
print("============================================================")

In [0]:
# ============================================================
# GRAVAÇÃO GOLD - FOOD ESTOQUE LOJAS
# REGRAS 6 A 10
# Upsert lógico via LEFT ANTI + UNION
# ============================================================


# ============================================================
# REGRA 6
# KPI: Taxa de ruptura de estoque por loja por mês
# Chave: id_loja + mes_snapshot
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_kpi_ruptura_estoque_mes"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_kpi_ruptura_estoque_mes

    print("Gold da Regra 6 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_kpi_ruptura_estoque_mes
        .select(
            "id_loja",
            "mes_snapshot"
        )
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on=[
                "id_loja",
                "mes_snapshot"
            ],
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_kpi_ruptura_estoque_mes
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 6 já existe.")
    print("Meses/lojas afetados foram recalculados.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_kpi_ruptura_estoque_mes"])
)

print("✅ Gold da Regra 6 gravada com sucesso.")


# ============================================================
# REGRA 7
# KPI: % de snapshots de perecíveis vinculados a lote vencido
# por loja
# Chave: id_loja + mes_snapshot
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_kpi_lotes_vencidos_por_loja"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_kpi_lotes_vencidos_por_loja

    print("Gold da Regra 7 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_kpi_lotes_vencidos_por_loja

        .select(
            "id_loja",
            "mes_snapshot"
        )
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on=[
                "id_loja",
                "mes_snapshot"
            ],
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_kpi_lotes_vencidos_por_loja

        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 7 já existe.")
    print("Meses/lojas afetados foram recalculados.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_kpi_lotes_vencidos_por_loja"])
)

print("✅ Gold da Regra 7 gravada com sucesso.")


# ============================================================
# REGRA 8
# KPI: Giro médio de estoque por categoria_pai
# Chave: tipo_produto + categoria_pai
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_giro_estoque_por_categoria_pai"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_giro_estoque_por_categoria_pai

    print("Gold da Regra 8 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_giro_estoque_por_categoria_pai
        .select(
            "tipo_produto",
            "categoria_pai"
        )
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on=[
                "tipo_produto",
                "categoria_pai"
            ],
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_giro_estoque_por_categoria_pai
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 8 já existe.")
    print("Categorias afetadas foram recalculadas.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_giro_estoque_por_categoria_pai"])
)

print("✅ Gold da Regra 8 gravada com sucesso.")


# ============================================================
# REGRA 9
# ALERTA: quantidade_disponivel negativa que não foi
# sinalizada pelo Silver
# Chave: id_loja + sku + id_lote + dt_snapshot
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_alerta_estoque_negativo"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_alerta_estoque_negativo

    print("Gold da Regra 9 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_alerta_estoque_negativo
        .select(
            "id_loja",
            "sku",
            "id_lote",
            "dt_snapshot"
        )
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on=[
                "id_loja",
                "sku",
                "id_lote",
                "dt_snapshot"
            ],
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_alerta_estoque_negativo
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 9 já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_alerta_estoque_negativo"])
)

print("✅ Gold da Regra 9 gravada com sucesso.")


# ============================================================
# REGRA 10
# Enriquecimento: ruptura de estoque coincide com queda nas vendas
# Chave: id_loja + sku + dt_snapshot
# ============================================================

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["gold_ruptura_estoque_enriquecida"])
    )

    df_gold_atual.limit(1).collect()
    gold_existe = True

except Exception:
    gold_existe = False


if not gold_existe:

    df_gold_final = df_gold_regra10

    print("Gold da Regra 10 não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:

    chaves_gold = (
        df_gold_regra10
        .select(
            "id_loja",
            "sku",
            "dt_snapshot"
        )
        .dropDuplicates()
    )

    df_gold_mantidas = (
        df_gold_atual
        .join(
            chaves_gold,
            on=[
                "id_loja",
                "sku",
                "dt_snapshot"
            ],
            how="left_anti"
        )
    )

    df_gold_final = (
        df_gold_regra10
        .unionByName(df_gold_mantidas)
    )

    print("Gold da Regra 10 já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .save(paths["gold_ruptura_estoque_enriquecida"])
)

print("✅ Gold da Regra 10 gravada com sucesso.")


print("============================================================")
print("✅ REGRAS GOLD DE ESTOQUE 6 A 10 GRAVADAS COM SUCESSO.")
print("============================================================")

In [0]:
df_gold_alerta_estoque_negativo.filter(
    col("id_loja").isNull() |
    col("sku").isNull() |
    col("id_lote").isNull() |
    col("dt_snapshot").isNull()
).count()

df_gold_giro_estoque_por_categoria_pai.filter(
    col("tipo_produto").isNull() |
    col("categoria_pai").isNull()
).count()

In [0]:
# ============================================================
# GRAVAÇÃO FOOD_AVALIACOES_PRODUTO
# GOLD -> SQL SERVER
#
# Serverless / Spark Connect
#
# IMPORTANTE:
# - CREATE / MERGE / DELETE / DROP:
#       spark.read.format("jdbc")
#
# - STAGING:
#       df.write.format("sqlserver")
# ============================================================


# ============================================================
# FOOD_AVALIACOES_PRODUTO - REGRA 6
# AVALIAÇÃO ANTES DO PEDIDO
#
# DataFrame:
#     df_gold_avaliacoes_antes_pedido
#
# Chave:
#     id_avaliacao
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_avaliacoes_antes_pedido', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_avaliacoes_antes_pedido (
        id_avaliacao INT NOT NULL,
        id_pedido INT,
        verificada BIT,
        dt_avaliacao DATETIME2(3),
        flag_avaliacao_antes_pedido BIT NOT NULL,
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_avaliacoes_antes_pedido
            PRIMARY KEY (id_avaliacao)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_avaliacoes_antes_pedido"

(
    df_gold_avaliacoes_antes_pedido
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_avaliacoes_antes_pedido AS destino
USING squad3.stg_gold_avaliacoes_antes_pedido AS origem

ON destino.id_avaliacao = origem.id_avaliacao

WHEN MATCHED THEN
    UPDATE SET
        destino.id_pedido = origem.id_pedido,
        destino.verificada = origem.verificada,
        destino.dt_avaliacao = origem.dt_avaliacao,
        destino.flag_avaliacao_antes_pedido = origem.flag_avaliacao_antes_pedido,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        id_avaliacao,
        id_pedido,
        verificada,
        dt_avaliacao,
        flag_avaliacao_antes_pedido,
        gold_ingested_at
    )
    VALUES (
        origem.id_avaliacao,
        origem.id_pedido,
        origem.verificada,
        origem.dt_avaliacao,
        origem.flag_avaliacao_antes_pedido,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_avaliacoes_antes_pedido', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_avaliacoes_antes_pedido;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_AVALIACOES_PRODUTO - REGRA 7
# KPI DE NOTA MÉDIA POR SKU E MÊS
#
# DataFrame:
#     df_gold_kpi_nota_sku_mes
#
# Chave:
#     sku + mes_avaliacao
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_kpi_nota_sku_mes', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_kpi_nota_sku_mes (
        sku NVARCHAR(255) NOT NULL,
        mes_avaliacao DATE NOT NULL,
        nota_media DECIMAL(18,6),
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_kpi_nota_sku_mes
            PRIMARY KEY (sku, mes_avaliacao)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_kpi_nota_sku_mes"

(
    df_gold_kpi_nota_sku_mes
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_kpi_nota_sku_mes AS destino
USING squad3.stg_gold_kpi_nota_sku_mes AS origem

ON destino.sku = origem.sku
AND destino.mes_avaliacao = origem.mes_avaliacao

WHEN MATCHED THEN
    UPDATE SET
        destino.nota_media = origem.nota_media,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        sku,
        mes_avaliacao,
        nota_media,
        gold_ingested_at
    )
    VALUES (
        origem.sku,
        origem.mes_avaliacao,
        origem.nota_media,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_kpi_nota_sku_mes', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_kpi_nota_sku_mes;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_AVALIACOES_PRODUTO - REGRA 8
# KPI DE AVALIAÇÕES POR MÊS
#
# DataFrame:
#     df_gold_kpi_avaliacoes_mes
#
# Chave:
#     mes_avaliacao
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_kpi_avaliacoes_mes', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_kpi_avaliacoes_mes (
        mes_avaliacao DATE NOT NULL,
        qtd_avaliacoes BIGINT NOT NULL,
        qtd_avaliacoes_verificadas BIGINT,
        qtd_avaliacoes_nao_verificadas BIGINT,
        pct_avaliacoes_verificadas DECIMAL(18,2),
        pct_avaliacoes_nao_verificadas DECIMAL(18,2),
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_kpi_avaliacoes_mes
            PRIMARY KEY (mes_avaliacao)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_kpi_avaliacoes_mes"

(
    df_gold_kpi_avaliacoes_mes
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_kpi_avaliacoes_mes AS destino
USING squad3.stg_gold_kpi_avaliacoes_mes AS origem

ON destino.mes_avaliacao = origem.mes_avaliacao

WHEN MATCHED THEN
    UPDATE SET
        destino.qtd_avaliacoes = origem.qtd_avaliacoes,
        destino.qtd_avaliacoes_verificadas = origem.qtd_avaliacoes_verificadas,
        destino.qtd_avaliacoes_nao_verificadas = origem.qtd_avaliacoes_nao_verificadas,
        destino.pct_avaliacoes_verificadas = origem.pct_avaliacoes_verificadas,
        destino.pct_avaliacoes_nao_verificadas = origem.pct_avaliacoes_nao_verificadas,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        mes_avaliacao,
        qtd_avaliacoes,
        qtd_avaliacoes_verificadas,
        qtd_avaliacoes_nao_verificadas,
        pct_avaliacoes_verificadas,
        pct_avaliacoes_nao_verificadas,
        gold_ingested_at
    )
    VALUES (
        origem.mes_avaliacao,
        origem.qtd_avaliacoes,
        origem.qtd_avaliacoes_verificadas,
        origem.qtd_avaliacoes_nao_verificadas,
        origem.pct_avaliacoes_verificadas,
        origem.pct_avaliacoes_nao_verificadas,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_kpi_avaliacoes_mes', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_kpi_avaliacoes_mes;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_AVALIACOES_PRODUTO - REGRA 9
# TOP 10 PIORES SKUS POR TRIMESTRE
#
# DataFrame:
#     df_gold_kpi_top10_piores_skus
#
# Chave lógica:
#     sku + trimestre_avaliacao
#
# Estratégia:
#     DELETE DOS TRIMESTRES AFETADOS + INSERT
#
# Motivo:
#     Um SKU pode sair do Top 10 em uma nova execução.
#     Portanto, não basta MERGE pela chave sku + trimestre.
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_kpi_top10_piores_skus', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_kpi_top10_piores_skus (
        sku NVARCHAR(255) NOT NULL,
        trimestre_avaliacao DATE NOT NULL,
        qtd_avaliacoes BIGINT NOT NULL,
        nota_media DECIMAL(18,6),
        ranking INT NOT NULL,
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_kpi_top10_piores_skus
            PRIMARY KEY (sku, trimestre_avaliacao)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_kpi_top10_piores_skus"

(
    df_gold_kpi_top10_piores_skus
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. DELETE DOS TRIMESTRES AFETADOS
# ============================================================

sql_delete = """
DELETE destino
FROM squad3.gold_kpi_top10_piores_skus AS destino
INNER JOIN (
    SELECT DISTINCT
        trimestre_avaliacao
    FROM squad3.stg_gold_kpi_top10_piores_skus
) AS origem
    ON destino.trimestre_avaliacao = origem.trimestre_avaliacao;
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_delete)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. INSERT DOS DADOS ATUALIZADOS
# ============================================================

sql_insert = """
INSERT INTO squad3.gold_kpi_top10_piores_skus (
    sku,
    trimestre_avaliacao,
    qtd_avaliacoes,
    nota_media,
    ranking,
    gold_ingested_at
)
SELECT
    origem.sku,
    origem.trimestre_avaliacao,
    origem.qtd_avaliacoes,
    origem.nota_media,
    origem.ranking,
    origem.gold_ingested_at
FROM squad3.stg_gold_kpi_top10_piores_skus AS origem;
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_insert)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 5. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_kpi_top10_piores_skus', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_kpi_top10_piores_skus;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_AVALIACOES_PRODUTO - REGRA 10
# AVALIAÇÕES ENRIQUECIDAS COM LOTE E FORNECEDOR
#
# DataFrame:
#     df_gold_avaliacoes_enriquecidas
#
# Chave:
#     id_avaliacao + id_lote
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# IMPORTANTE
#
# Como o enriquecimento utiliza LEFT JOIN com lotes,
# id_lote pode ser NULL.
#
# Se id_lote estiver NULL no DataFrame, essa chave não poderá
# ser PRIMARY KEY no SQL Server.
#
# Antes de executar, validar:
#
# df_gold_avaliacoes_enriquecidas.filter(
#     col("id_avaliacao").isNull() |
#     col("id_lote").isNull()
# ).count()
#
# O ideal é esse resultado ser 0 antes de criar a PK.
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_avaliacoes_enriquecidas', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_avaliacoes_enriquecidas (
        id_avaliacao INT NOT NULL,
        id_pedido INT,
        sku NVARCHAR(255),
        nota INT,
        dt_avaliacao DATETIME2(3),
        verificada BIT,
        id_lote INT NOT NULL,
        id_fornecedor INT,
        fornecedor NVARCHAR(255),
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_avaliacoes_enriquecidas
            PRIMARY KEY (id_avaliacao, id_lote)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_avaliacoes_enriquecidas"

(
    df_gold_avaliacoes_enriquecidas
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_avaliacoes_enriquecidas AS destino
USING squad3.stg_gold_avaliacoes_enriquecidas AS origem

ON destino.id_avaliacao = origem.id_avaliacao
AND destino.id_lote = origem.id_lote

WHEN MATCHED THEN
    UPDATE SET
        destino.id_pedido = origem.id_pedido,
        destino.sku = origem.sku,
        destino.nota = origem.nota,
        destino.dt_avaliacao = origem.dt_avaliacao,
        destino.verificada = origem.verificada,
        destino.id_fornecedor = origem.id_fornecedor,
        destino.fornecedor = origem.fornecedor,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        id_avaliacao,
        id_pedido,
        sku,
        nota,
        dt_avaliacao,
        verificada,
        id_lote,
        id_fornecedor,
        fornecedor,
        gold_ingested_at
    )
    VALUES (
        origem.id_avaliacao,
        origem.id_pedido,
        origem.sku,
        origem.nota,
        origem.dt_avaliacao,
        origem.verificada,
        origem.id_lote,
        origem.id_fornecedor,
        origem.fornecedor,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_avaliacoes_enriquecidas', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_avaliacoes_enriquecidas;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)

In [0]:
# ============================================================
# GRAVAÇÃO FOOD_ESTOQUE_LOJAS
# GOLD -> SQL SERVER
#
# Serverless / Spark Connect
#
# IMPORTANTE:
# - CREATE / MERGE / DROP: spark.read.format("jdbc")
# - STAGING: df.write.format("sqlserver")
# ============================================================


# ============================================================
# FOOD_ESTOQUE_LOJAS - REGRA 6
# KPI DE RUPTURA DE ESTOQUE POR LOJA E MÊS
#
# DataFrame:
#     df_gold_kpi_ruptura_estoque_mes
#
# Chave:
#     id_loja + mes_snapshot
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_kpi_ruptura_estoque', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_kpi_ruptura_estoque (
        id_loja INT NOT NULL,
        nome_loja NVARCHAR(255),
        mes_snapshot DATE NOT NULL,
        qtd_snapshots BIGINT NOT NULL,
        qtd_snapshots_ruptura BIGINT,
        taxa_ruptura_pct DECIMAL(18,2),
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_kpi_ruptura_estoque
            PRIMARY KEY (id_loja, mes_snapshot)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_kpi_ruptura_estoque"

(
    df_gold_kpi_ruptura_estoque_mes
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_kpi_ruptura_estoque AS destino
USING squad3.stg_gold_kpi_ruptura_estoque AS origem

ON destino.id_loja = origem.id_loja
AND destino.mes_snapshot = origem.mes_snapshot

WHEN MATCHED THEN
    UPDATE SET
        destino.nome_loja = origem.nome_loja,
        destino.qtd_snapshots = origem.qtd_snapshots,
        destino.qtd_snapshots_ruptura = origem.qtd_snapshots_ruptura,
        destino.taxa_ruptura_pct = origem.taxa_ruptura_pct,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        id_loja,
        nome_loja,
        mes_snapshot,
        qtd_snapshots,
        qtd_snapshots_ruptura,
        taxa_ruptura_pct,
        gold_ingested_at
    )
    VALUES (
        origem.id_loja,
        origem.nome_loja,
        origem.mes_snapshot,
        origem.qtd_snapshots,
        origem.qtd_snapshots_ruptura,
        origem.taxa_ruptura_pct,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_kpi_ruptura_estoque', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_kpi_ruptura_estoque;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_ESTOQUE_LOJAS - REGRA 7
# KPI DE LOTES VENCIDOS POR LOJA E MÊS
#
# DataFrame:
#     df_gold_kpi_lotes_vencidos_por_loja
#
# Chave:
#     id_loja + mes_snapshot
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_kpi_lotes_vencidos', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_kpi_lotes_vencidos (
        id_loja INT NOT NULL,
        nome_loja NVARCHAR(255),
        mes_snapshot DATE NOT NULL,
        qtd_snapshots BIGINT NOT NULL,
        qtd_snapshots_lote_vencido BIGINT,
        pct_snapshots_lote_vencido DECIMAL(18,2),
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_kpi_lotes_vencidos
            PRIMARY KEY (id_loja, mes_snapshot)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_kpi_lotes_vencidos"

(
    df_gold_kpi_lotes_vencidos_por_loja
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_kpi_lotes_vencidos AS destino
USING squad3.stg_gold_kpi_lotes_vencidos AS origem

ON destino.id_loja = origem.id_loja
AND destino.mes_snapshot = origem.mes_snapshot

WHEN MATCHED THEN
    UPDATE SET
        destino.nome_loja = origem.nome_loja,
        destino.qtd_snapshots = origem.qtd_snapshots,
        destino.qtd_snapshots_lote_vencido = origem.qtd_snapshots_lote_vencido,
        destino.pct_snapshots_lote_vencido = origem.pct_snapshots_lote_vencido,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        id_loja,
        nome_loja,
        mes_snapshot,
        qtd_snapshots,
        qtd_snapshots_lote_vencido,
        pct_snapshots_lote_vencido,
        gold_ingested_at
    )
    VALUES (
        origem.id_loja,
        origem.nome_loja,
        origem.mes_snapshot,
        origem.qtd_snapshots,
        origem.qtd_snapshots_lote_vencido,
        origem.pct_snapshots_lote_vencido,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_kpi_lotes_vencidos', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_kpi_lotes_vencidos;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_ESTOQUE_LOJAS - REGRA 8
# GIRO MÉDIO DE ESTOQUE POR TIPO DE PRODUTO E CATEGORIA PAI
#
# DataFrame:
#     df_gold_giro_estoque_por_categoria_pai
#
# Chave:
#     tipo_produto + categoria_pai
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_kpi_giro_medio_estoque', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_kpi_giro_medio_estoque (
        tipo_produto NVARCHAR(255) NOT NULL,
        categoria_pai NVARCHAR(255) NOT NULL,
        giro_medio_dias DECIMAL(18,6),
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_kpi_giro_medio_estoque
            PRIMARY KEY (tipo_produto, categoria_pai)
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_kpi_giro_medio_estoque"

(
    df_gold_giro_estoque_por_categoria_pai
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_kpi_giro_medio_estoque AS destino
USING squad3.stg_gold_kpi_giro_medio_estoque AS origem

ON destino.tipo_produto = origem.tipo_produto
AND destino.categoria_pai = origem.categoria_pai

WHEN MATCHED THEN
    UPDATE SET
        destino.giro_medio_dias = origem.giro_medio_dias,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        tipo_produto,
        categoria_pai,
        giro_medio_dias,
        gold_ingested_at
    )
    VALUES (
        origem.tipo_produto,
        origem.categoria_pai,
        origem.giro_medio_dias,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_kpi_giro_medio_estoque', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_kpi_giro_medio_estoque;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_ESTOQUE_LOJAS - REGRA 9
# ALERTA DE ESTOQUE NEGATIVO
#
# DataFrame:
#     df_gold_alerta_estoque_negativo
#
# Chave:
#     id_loja + sku + id_lote + dt_snapshot
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_alerta_estoque_negativo', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_alerta_estoque_negativo (
        id_loja INT NOT NULL,
        sku NVARCHAR(255) NOT NULL,
        id_lote INT NOT NULL,
        quantidade_disponivel INT,
        dt_snapshot DATE NOT NULL,
        flag_quantidade_invalida BIT,
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_alerta_estoque_negativo
            PRIMARY KEY (
                id_loja,
                sku,
                id_lote,
                dt_snapshot
            )
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_alerta_estoque_negativo"

(
    df_gold_alerta_estoque_negativo
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_alerta_estoque_negativo AS destino
USING squad3.stg_gold_alerta_estoque_negativo AS origem

ON destino.id_loja = origem.id_loja
AND destino.sku = origem.sku
AND destino.id_lote = origem.id_lote
AND destino.dt_snapshot = origem.dt_snapshot

WHEN MATCHED THEN
    UPDATE SET
        destino.quantidade_disponivel = origem.quantidade_disponivel,
        destino.flag_quantidade_invalida = origem.flag_quantidade_invalida,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        id_loja,
        sku,
        id_lote,
        quantidade_disponivel,
        dt_snapshot,
        flag_quantidade_invalida,
        gold_ingested_at
    )
    VALUES (
        origem.id_loja,
        origem.sku,
        origem.id_lote,
        origem.quantidade_disponivel,
        origem.dt_snapshot,
        origem.flag_quantidade_invalida,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_alerta_estoque_negativo', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_alerta_estoque_negativo;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# FOOD_ESTOQUE_LOJAS - REGRA 10
# RUPTURA DE ESTOQUE COINCIDINDO COM QUEDA DE VENDAS
#
# DataFrame:
#     df_gold_ruptura_estoque_enriquecida
#
# Chave:
#     id_loja + sku + dt_snapshot
#
# Estratégia:
#     MERGE
# ============================================================


# ============================================================
# 1. CRIAR TABELA DEFINITIVA
# ============================================================

sql_create_table = """
IF OBJECT_ID('squad3.gold_regra10_ruptura_queda_vendas', 'U') IS NULL
BEGIN
    CREATE TABLE squad3.gold_regra10_ruptura_queda_vendas (
        id_loja INT NOT NULL,
        sku NVARCHAR(255) NOT NULL,
        dt_snapshot DATE NOT NULL,
        quantidade_disponivel INT,
        estoque_minimo INT,
        quantidade_vendida DECIMAL(18,6) NOT NULL,
        quantidade_vendida_anterior DECIMAL(18,6),
        flag_ruptura_coincide_queda_vendas BIT,
        gold_ingested_at DATETIME2(3) NOT NULL,

        CONSTRAINT pk_gold_regra10_ruptura_queda_vendas
            PRIMARY KEY (
                id_loja,
                sku,
                dt_snapshot
            )
    );
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_create_table)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 2. GRAVAR GOLD NA STAGING
# ============================================================

staging_table = "squad3.stg_gold_regra10_ruptura_queda_vendas"

(
    df_gold_ruptura_estoque_enriquecida
    .write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_database)
    .option("dbtable", staging_table)
    .option("user", jdbc_username)
    .option("password", jdbc_password)
    .mode("overwrite")
    .save()
)


# ============================================================
# 3. MERGE
# ============================================================

sql_merge = """
MERGE squad3.gold_regra10_ruptura_queda_vendas AS destino
USING squad3.stg_gold_regra10_ruptura_queda_vendas AS origem

ON destino.id_loja = origem.id_loja
AND destino.sku = origem.sku
AND destino.dt_snapshot = origem.dt_snapshot

WHEN MATCHED THEN
    UPDATE SET
        destino.quantidade_disponivel = origem.quantidade_disponivel,
        destino.estoque_minimo = origem.estoque_minimo,
        destino.quantidade_vendida = origem.quantidade_vendida,
        destino.quantidade_vendida_anterior = origem.quantidade_vendida_anterior,
        destino.flag_ruptura_coincide_queda_vendas = origem.flag_ruptura_coincide_queda_vendas,
        destino.gold_ingested_at = origem.gold_ingested_at

WHEN NOT MATCHED BY TARGET THEN
    INSERT (
        id_loja,
        sku,
        dt_snapshot,
        quantidade_disponivel,
        estoque_minimo,
        quantidade_vendida,
        quantidade_vendida_anterior,
        flag_ruptura_coincide_queda_vendas,
        gold_ingested_at
    )
    VALUES (
        origem.id_loja,
        origem.sku,
        origem.dt_snapshot,
        origem.quantidade_disponivel,
        origem.estoque_minimo,
        origem.quantidade_vendida,
        origem.quantidade_vendida_anterior,
        origem.flag_ruptura_coincide_queda_vendas,
        origem.gold_ingested_at
    );
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_merge)
    .options(**jdbc_properties)
    .load()
)


# ============================================================
# 4. REMOVER STAGING
# ============================================================

sql_drop_staging = """
IF OBJECT_ID('squad3.stg_gold_regra10_ruptura_queda_vendas', 'U') IS NOT NULL
BEGIN
    DROP TABLE squad3.stg_gold_regra10_ruptura_queda_vendas;
END
"""

(
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1")
    .option("sessionInitStatement", sql_drop_staging)
    .options(**jdbc_properties)
    .load()
)